In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.special import erfcx
from ana_rtd_comp import ana_rtd
from ana_acc_comp import ana_acc
import pandas as pd
from scipy.special import erfc
import seaborn as sns
import ast
from pybads import BADS
import numpy as np
import matplotlib.pyplot as plt
from model_tach_shift_v2 import model_tach_shift

tt=np.arange(0,1000, 1)
compressive_power= 0.2
a_tied= 20
T0= 2
ndt=70
v=0.12
a=60
abl=20
ild=2
t_fix= 400
w_tse= .75
tse = ndt*w_tse
Delta = -tse+10

x = [compressive_power, a_tied, T0, ndt, v, a, w_tse, Delta]

# model = model_tach_shift(ABL=abl,ILD=ild,t_fix=t_fix)
# initialize the model
model = model_tach_shift(RT=tt,ABL=abl*np.ones_like(tt),ILD=ild*np.ones_like(tt),t_fix=t_fix*np.ones_like(tt),out=1.0*np.ones_like(tt))
model.set_params(x)
model.print_params()


logL_corr, logL_err = model.log_likelihood_both()
print(np.sum(logL_corr)) 
print(np.sum(logL_err))

logL = model.log_likelihood()
print(np.sum(logL))

[ 0.2 20.   2.  70. ] [ 0.12 60.  ] 0.75 -42.5
-10327.143112095364
-10888.225271976557
-10327.143112095364


In [2]:
df= pd.read_csv('/Users/utsai/Desktop/short_dur/dow/out_SD.csv')
df['RT'] = df['timed_fix']-df['intended_fix']
df['RT_ms']= df['RT']*1000
df['intended_fix_ms']= df['intended_fix']*1000 
df['time_to_CNP_c'] = df['time_to_CNP'].apply(lambda x: 1 if x < 0.002 else 0)
df['timed_fix_ms']= df['timed_fix']*1000
df = df[
    (df['ABL'].isin([20, 40, 60])) &
    (df['ILD'].isin([-1, -2, -4, -8, 1, 2, 4, 8])) &
    (df['abort_event'].isin([-1,-2])) &
    (df['time_to_CNP_c'] == 0)&
    (df['max_samplingTime'].isin([6]))&
    (df['timed_fix_ms']>200)
    ]

In [3]:
x = [compressive_power, a_tied, T0, ndt, v, a, w_tse, Delta]
rt_vect=np.array(df['RT_ms'])
abl_vect=np.array(df['ABL'])
ild_vect=np.array(df['ILD'])
fix_vect=np.array(df['intended_fix_ms'])
out_vect=np.array(df['success'])
# model = model_tach_shift(ABL=abl,ILD=ild,t_fix=t_fix)
# initialize the model
model = model_tach_shift(RT=rt_vect,ABL=abl_vect,ILD=ild_vect,t_fix=fix_vect, out=out_vect)
model.set_params(x)
model.print_params()


logL_corr, logL_err = model.log_likelihood_both()
print(np.sum(logL_corr)) 
print(np.sum(logL_err))

logL = model.log_likelihood()
print(np.sum(logL))

[ 0.2 20.   2.  70. ] [ 0.12 60.  ] 0.75 -42.5
FS: [0.96491816 0.21140447 0.86322696 ... 0.99076462 0.96784746 0.96593624]
pdf:[3.75079233e-04 5.67292398e-03 6.37547866e-03 ... 8.75156638e-07
 4.90778202e-04 5.13258674e-05]
-297769.4280100746
-339584.6667711582
FS: [0.96491816 0.21140447 0.86322696 ... 0.99076462 0.96784746 0.96593624]
pdf:[3.75079233e-04 5.67292398e-03 6.37547866e-03 ... 8.75156638e-07
 4.90778202e-04 5.13258674e-05]
-305640.49267276697


In [3]:
def total_log_likelihood(data, params):
    rt_vect=np.array(data['RT_ms'])
    abl_vect=np.array(data['ABL'])
    ild_vect=np.array(data['ILD'])
    fix_vect=np.array(data['intended_fix_ms'])
    out_vect=np.array(data['success'])
    model = model_tach_shift(RT=rt_vect,ABL=abl_vect,ILD=ild_vect,t_fix=fix_vect, out=out_vect)
    model.set_params(params)
    logL = model.log_likelihood()
    return np.sum(logL)

In [5]:
def objective_function(params, data):
    """ 
    Compute the negative log-likelihood for the DDM parameters.
    
    :param params: array of model parameters [v, a, x1, x2, x3, x4]
    :param data: DataFrame with the experimental data
    :return: Negative log-likelihood

    """ # Assuming x includes 4 parameters
    x=params
    # Calculate the total log likelihood using the defined function
    total_likelihood = total_log_likelihood(data, x)
    # BADS minimizes the function, so return negative log likelihood
    return -total_likelihood
#creat a suitable function for bads

def objective_fun_forbads(params):
    data= df_animal
    return objective_function(params, data)

# x = [compressive_power, a_tied, T0, ndt, v, a, w_tse, Delta]
initial_param_sets = [
    [0.1, 35, 1, 40,0.08,20,0.2,-30],
    [0.2, 20, 0.5, 50,0.06,30,0.3,-20],
    [0.25, 30, 0.75,60, 0.1, 50,0.1,-10]
] 
lower_bounds = [0.03, 10, 0.05, 0,0 ,10, 0, -80]  # Hard lower bounds
upper_bounds = [0.5, 80, 2, 100, 1, 100, 1, 10]  # Hard upper bounds
plausible_lb = [0.04, 15, 0.1,10, 0.02,15, 0.01,-70]  # Plausible lower bounds
plausible_ub = [0.4, 60, 1.8, 80, 0.8, 80, 0.9,0]  # Plausible upper bounds

# Run optimization for each animal
all_results = []
animal_value= df['animal'].unique()
for idx, animal in enumerate(animal_value):
    print(animal)
    best_result = None
    df_animal = df[(df['animal'] == animal)&(df['max_samplingTime']==6)]
    row_count = len(df_animal)
    for i, initial_params in enumerate(initial_param_sets):
        print(f"Starting optimization with initial params: {initial_params}")
        bads = BADS(objective_fun_forbads, initial_params, lower_bounds, upper_bounds, plausible_lb, plausible_ub)
        optimize_result = bads.optimize()
        result_details = {
            'animal': animal,
            'initial_params': str(initial_params),
            'optimized_params': str(optimize_result.x),
            'result_fun': optimize_result.fval,
            'row_count': row_count,
            'is_best': 0  # Default to 0, will update later if it's the best
        }
        print(f"Optimization result: {optimize_result.x}, {optimize_result.fval}")
        all_results.append(result_details)

        # Update the best result if it's the first run or better than the previous best
        if best_result is None or optimize_result.fval < best_result.fval:
            best_result = optimize_result
            best_params = initial_params

    # Update the best result indicator for the best optimization run
    for result in all_results:
        if result['animal'] == animal and result['initial_params'] == str(best_params):
            result['is_best'] = 1

# Convert all_results to a DataFrame
results_df = pd.DataFrame(all_results)
results_df.to_csv('tied_pa_accu_fit_7params.csv', index=False)

48
Starting optimization with initial params: [0.1, 35, 1, 40, 0.08, 20, 0.2, -30]
Variables (index) internally transformed to log coordinates: [[0 0]
 [0 2]]
Beginning optimization of a DETERMINISTIC objective function

 Iteration    f-count         f(x)           MeshScale          Method             Actions
     0           2         44900.2               1                                 Uncertainty test
     0          18         40703.5               1         Initial mesh            Initial points
     0          27         36231.6               1       Successful poll           Train
     1          31         36032.4               1     Successful search (ES-wcm)        
     1          34         35861.8               1     Successful search (ES-wcm)        
     1          41         35853.5               1     Successful search (ES-ell)        
     1          43         35784.9               1     Successful search (ES-wcm)        
     1          44         35718.9       

KeyboardInterrupt: 